# Introduction

Cart abandonment is a critical challenge for digital ordering platforms, directly impacting revenue and customer retention. For MyCoke360, Coca-Cola's B2B digital ordering system launched in Summer 2024, understanding why customers fail to complete purchases is especially important. The platform serves Food Service On Premise (FSOP) customers such as restaurants, schools, hospitals, and retailers, where order frequency and product mix drive significant business value. By examining customer behavior captured in Google Analytics alongside order and sales data, this project seeks to uncover patterns that explain when, how, and why carts are abandoned.

This exploratory data analysis (EDA) will focus on evaluating the quality, structure, and usability of the available data so that it is ready to be used for financial evaluation modeling in the later modeling stage. Other aspects of the problem statement such as identifying behavioral predictors, analyzing recovery patterns, and evaluating device-specific abandonment will be addressed by other members of the project team. This division of workflow ensures comprehensive coverage of the problem space while allowing each stage of the analysis to build on a solid data foundation.

## Initial Guiding Questions

- What is the most efficient method for representing the previously defined cart abondonment within the data?
- How can abandoned carts be aggregated to accurately estimate lost revenue at both the order and product level?
- Which product categories, pack types, or SKUs appear most frequently in abandoned carts, and how should these be visualized for clear insights?
- What is the distribution of abandonment across different customer segments, such as sales office, plant, or FSOP type?
- How can abandoned cart revenue be compared against total sales to highlight the relative financial impact?
- What temporal patterns emerge in abandoned carts (e.g., by day of week, order cycle, or over time during the study period)?
- Are there systematic differences in abandonment linked to operational factors such as cutoff times or anchor days?

While these guiding questions move closer to addressing the main problem of the project, the exploratory data analysis may not answer them directly. Instead, they will serve to guide how the data is structured, organized, and prepared so that later modeling and analysis can properly evaluate the impact of cart abandonment.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import json
import re

# Initial Data Evaluation

The dataset consists of eight CSV tables covering customer behavior, transactions, and supporting reference information. Three fact tables capture activity on MyCoke360: Google Analytics events (site visits, add/remove cart actions, purchases, and device/page details), Orders (materials ordered per customer, with order type and timestamps in both EST and UTC), and Sales (fulfilled transactions with pricing and profit measures). These are complemented by five dimension tables: Customer (account and channel attributes, sales office details), Cutoff Times (order cutoff policies by plant, office, and distribution mode), Material (product master data such as pack type, brand, flavor, and category), Operating Hours (current ordering frequency and anchor day/date by customer), and Visit Plan (historical anchor dates, frequencies, and sales office attributes). Collectively, these tables create a comprehensive view of both customer behavior and business processes adequetely enabling our analysis.

## Helper Functions

In [0]:
def summarize(df: pd.DataFrame, name: str = "DataFrame"):
    """
    Rough equivalent of the Spark .summary() + custom null/blank/'null' counts.
    Prints a describe(include='all') and a 'nulls' row that counts:
      - NaN/None
      - empty strings (after trim)
      - literal case-insensitive 'null' strings
    """
    print(f"\n=== Summary for: {name} ===")
    # Basic stats
    try:
        desc = df.describe(include='all', datetime_is_numeric=True)
    except Exception:
        # Fallback for older pandas
        desc = df.describe(include='all')
    print(desc)

    # Null-like counts
    null_counts = {}
    for col in df.columns:
        s = df[col]
        # Start with isna
        is_null = s.isna()
        # Empty strings (only for string-like)
        try:
            is_empty = s.astype(str).str.strip().eq("")
        except Exception:
            is_empty = pd.Series(False, index=s.index)
        # Literal "null" (case-insensitive)
        try:
            is_lit_null = s.astype(str).str.strip().str.lower().eq("null")
        except Exception:
            is_lit_null = pd.Series(False, index=s.index)

        null_counts[col] = (is_null | is_empty | is_lit_null).sum()

    nulls_row = pd.DataFrame([null_counts], index=["nulls"])
    print("\nNull-like counts (NaN | '' | 'null'):\n", nulls_row)


def read_csv(path: str) -> pd.DataFrame:
    """
    Read CSV similarly to the Spark options used.
    """
    return pd.read_csv(
        path,
        header=0,
        quotechar='"',
        escapechar='"',
        dtype=str,          # match inferSchema=False
        low_memory=False
    )


def parse_items_column(series: pd.Series) -> pd.Series:
    """
    Parse the ITEMS column which is a JSON list of objects like:
      [{"item_id": "123", "quantity": "2"}, ...]
    Return a Python object (list of dicts) per row, or NaN on failure.
    """
    def _parse(val):
        if pd.isna(val):
            return np.nan
        txt = str(val)
        # Clean obvious doubled-quotes if present (from CSV escaping)
        # but keep JSON valid
        # We'll try a couple of strategies
        for candidate in (txt, txt.replace('""', '"')):
            try:
                obj = json.loads(candidate)
                # Coerce to ints where possible
                if isinstance(obj, list):
                    for d in obj:
                        if isinstance(d, dict):
                            if "item_id" in d:
                                try:
                                    d["item_id"] = int(d["item_id"])
                                except Exception:
                                    pass
                            if "quantity" in d:
                                try:
                                    d["quantity"] = int(d["quantity"])
                                except Exception:
                                    pass
                return obj
            except Exception:
                continue
        return np.nan

    return series.apply(_parse)


def to_numeric_strip_commas(series: pd.Series) -> pd.Series:
    if series is None:
        return series
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False), errors="coerce")


def coalesce_series(*series_list: pd.Series) -> pd.Series:
    """
    Return the first non-null entry across the provided series (row-wise).
    Equivalent of Spark coalesce.
    """
    out = series_list[0].copy()
    for s in series_list[1:]:
        out = out.where(out.notna(), s)
    return out

## Data Importing

In [0]:
# # Import fact tables
# google_analytics_raw = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/google_analytics.csv",
#     header=True,
#     inferSchema=False,
#     quote='"',
#     escape='"',
#     multiLine=True
# )
# orders_raw = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/orders.csv",
#     header=True,
#     inferSchema=False,
#     quote='"',
#     escape='"',
#     multiLine=True
# )
# sales_raw = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/sales.csv",
#     header=True,
#     inferSchema=False,
#     quote='"',
#     escape='"',
#     multiLine=True
# )

# # Import dimension tables
# customers_raw = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/customer.csv",
#     header=True,
#     inferSchema=False,
#     quote='"',
#     escape='"',
#     multiLine=True
# )
# materials_raw = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/material.csv",
#     header=True,
#     inferSchema=False,
#     quote='"',
#     escape='"',
#     multiLine=True
# )
# cutoff_times_raw = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/cutoff_times.csv",
#     header=True,
#     inferSchema=False,
#     quote='"',
#     escape='"',
#     multiLine=True
# )
# operating_hours_raw = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/operating_hours.csv",
#     header=True,
#     inferSchema=False,
#     quote='"',
#     escape='"',
#     multiLine=True
# )
# visit_plan_raw = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/visit_plan.csv",
#     header=True,
#     inferSchema=False,
#     quote='"',
#     escape='"',
#     multiLine=True
# )

# Import fact tables
google_analytics_raw = pd.read_csv(
    "/Volumes/workspace/default/capstone_data/google_analytics.csv",
    dtype=str,
    low_memory=False,
    # engine="python"
)
orders_raw = pd.read_csv(
    "/Volumes/workspace/default/capstone_data/orders.csv",
    dtype=str,
    low_memory=False,
    # engine="python"
)
sales_raw = pd.read_csv(
    "/Volumes/workspace/default/capstone_data/sales.csv",
    dtype=str,
    low_memory=False,
    # engine="python"
)

# Import dimension tables
customers_raw = pd.read_csv(
    "/Volumes/workspace/default/capstone_data/customer.csv",
    dtype=str,
    low_memory=False,
    # engine="python"
)
materials_raw = pd.read_csv(
    "/Volumes/workspace/default/capstone_data/material.csv",
    dtype=str,
    low_memory=False,
    # engine="python"
)
cutoff_times_raw = pd.read_csv(
    "/Volumes/workspace/default/capstone_data/cutoff_times.csv",
    dtype=str,
    low_memory=False,
    # engine="python"
)
operating_hours_raw = pd.read_csv(
    "/Volumes/workspace/default/capstone_data/operating_hours.csv",
    dtype=str,
    low_memory=False,
    # engine="python"
)
visit_plan_raw = pd.read_csv(
    "/Volumes/workspace/default/capstone_data/visit_plan.csv",
    dtype=str,
    low_memory=False,
    # engine="python"
)

## Google Analytics

In [0]:
# print(google_analytics_raw.head())
# summarize(google_analytics_raw, name="google_analytics_raw")

display(google_analytics_raw.head(5))
display(summarize(google_analytics_raw))

The dataset requires several data type adjustments to ensure proper analysis. EVENT_DATE should be cast to a date format, while EVENT_TIMESTAMP needs to be converted to a UTC datetime. The ITEMS field should be stored as an array of objects to capture item-level details more effectively. Data quality checks reveal that DEVICE_MOBILE_BRAND_NAME has 39,468 null values, which is expected given the mix of mobile, desktop, and tablet device types. EVENT_PAGE_NAME contains 1,002,510 null values and EVENT_PAGE_TITLE has 318,091 null values. Both issues are likely due to Google Analytics limitations, and removing these records could lead to biased or corrupted results. Finally, as noted in the project description, the ITEMS arrays are empty for mobile device records when they should contain item details. To resolve this, item information will need to be pulled from the orders table and joined back into the dataset.

Observations:
- EVENT_DATE needs to be cast to a date.
- EVENT_TIMESTAMP needs to be cast to a UTC datetime.
- ITEMS needs to be cast to an array of objects.
- DEVICE_MOBILE_BRAND_NAME has 39468 null values, though likely due to the data mix between mobile, desktop and tablet device types.
- EVENT_PAGE_NAME has 1002510 null values. This is likely due to limited Google Analytics definitions. Removing nulls might corrupt latter results.
- EVENT_PAGE_TITLE has 318091 null values. This is likely due to limited Google Analytics definitions. Removing nulls might corrupt latter results.
- As noted in the project description, the ITEMS arrays are empty for mobile devices when there should be items in some of them. This will need the items moved over from the orders table.

In [0]:
google_analytics = google_analytics_raw.copy()

# Parse dates/timestamps
google_analytics["EVENT_DATE"] = pd.to_datetime(google_analytics["EVENT_DATE"], format="%Y-%m-%d", errors="coerce")
google_analytics["EVENT_TIMESTAMP_UTC"] = pd.to_datetime(google_analytics["EVENT_TIMESTAMP"], errors="coerce", utc=True)

# Parse ITEMS JSON column
google_analytics["ITEMS"] = parse_items_column(google_analytics.get("ITEMS"))

# Select / order columns
google_analytics = google_analytics[[
    "CUSTOMER_ID",
    "EVENT_TIMESTAMP_UTC",
    "EVENT_NAME",
    "DEVICE_CATEGORY",
    "DEVICE_MOBILE_BRAND_NAME",
    "DEVICE_OPERATING_SYSTEM",
    "EVENT_PAGE_NAME",
    "EVENT_PAGE_TITLE",
    "ITEMS"
]].sort_values(["CUSTOMER_ID", "EVENT_TIMESTAMP_UTC"], kind="mergesort")

This code applies the data type corrections and prepares the table for analysis. EVENT_DATE is cast to a proper date. EVENT_TIMESTAMP is parsed into a new UTC-oriented timestamp column named EVENT_TIMESTAMP_UTC, aligning with the need for consistent time handling. ITEMS is converted from a JSON string into an array of objects with item_id and quantity stored as integers, which enables reliable item-level aggregation.

Columns with high null rates are preserved rather than filtered, which avoids introducing bias from Google Analytics field sparsity. Should a specific analysis require the null values to be removed, they will be at that time. The final projection keeps only the fields needed for analysis and orders the rows by customer and timestamp, which helps with downstream sequencing and sessionization tasks.

The code does not yet repair empty item arrays for mobile events. That backfill will come from joining in item details from the orders table in a later step.

In [0]:
# Events by month
events_of_interest = {"purchase", "add_to_cart", "remove_from_cart"}
ga_eoi = google_analytics[google_analytics["EVENT_NAME"].isin(events_of_interest)].copy()
ga_eoi["YEAR"] = ga_eoi["EVENT_TIMESTAMP_UTC"].dt.year
ga_eoi["MONTH"] = ga_eoi["EVENT_TIMESTAMP_UTC"].dt.month
ga_eoi["YEAR_MONTH"] = ga_eoi["EVENT_TIMESTAMP_UTC"].dt.strftime("%Y-%m")

events_by_month = (
    ga_eoi.groupby(["YEAR", "MONTH", "EVENT_NAME"], dropna=False)
    .size()
    .reset_index(name="count")
)
events_by_month["YEAR_MONTH"] = events_by_month.apply(
    lambda r: f"{int(r['YEAR']):04d}-{int(r['MONTH']):02d}", axis=1
)

# removed_or_purchased aggregation
rop = (
    events_by_month[events_by_month["EVENT_NAME"].isin(["purchase", "remove_from_cart"])]
    .groupby(["YEAR", "MONTH", "YEAR_MONTH"], as_index=False)["count"]
    .sum()
)
rop["EVENT_NAME"] = "removed_or_purchased"

evm = events_by_month[["YEAR_MONTH", "EVENT_NAME", "count"]]
events_by_month_all = pd.concat([evm, rop[["YEAR_MONTH", "EVENT_NAME", "count"]]], ignore_index=True)
events_by_month_all = events_by_month_all.sort_values("YEAR_MONTH")

events_pivot = (
    events_by_month_all
    .pivot(index="YEAR_MONTH", columns="EVENT_NAME", values="count")
    .fillna(0)
    .sort_index()
)
events_pivot.index = pd.to_datetime(events_pivot.index, format="%Y-%m")

# Plot
plt.figure(figsize=(10,6))
order = ["add_to_cart", "removed_or_purchased", "remove_from_cart", "purchase"]
for colname in order:
    if colname in events_pivot.columns:
        plt.plot(events_pivot.index, events_pivot[colname], marker="o", label=colname)

plt.title("Monthly Transaction Events Over Time")
plt.xlabel("Month")
plt.ylabel("Count")

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xticks(rotation=45)
plt.legend(title="Event Type")
plt.grid(False)
plt.tight_layout()
plt.show()

The above line graph compares monthly counts of Add to Cart, Remove from Cart, and Purchase events. Across all periods, the number of Add to Cart actions is consistently higher than the combined totals of Purchases and Remove from Cart events. This gap indicates that many items added to carts are not being acted upon, either purchased or explicitly removed. The persistent surplus of Add to Cart activity suggests signs of cart bloating and potential cart abandonment since we would expect the Add to Cart counts to closely match the sum of Purchases and Removals if items were being consistently finalized.

In [0]:
# Device category / OS / brand counts (Top 10)
device_category_counts = (
    google_analytics["DEVICE_CATEGORY"].value_counts(dropna=False).reset_index()
    .rename(columns={"index": "DEVICE_CATEGORY", "DEVICE_CATEGORY": "count"})
    .head(10)
)

device_os_counts = (
    google_analytics["DEVICE_OPERATING_SYSTEM"].value_counts(dropna=False).reset_index()
    .rename(columns={"index": "DEVICE_OPERATING_SYSTEM", "DEVICE_OPERATING_SYSTEM": "count"})
    .head(10)
)

device_brand_counts = (
    google_analytics["DEVICE_MOBILE_BRAND_NAME"].value_counts(dropna=False).reset_index()
    .rename(columns={"index": "DEVICE_MOBILE_BRAND_NAME", "DEVICE_MOBILE_BRAND_NAME": "count"})
    .head(10)
)

# Horizontal bar charts with numeric y + custom labels
fig, axes = plt.subplots(1, 3, figsize=(12, 6))

# Device category
y0 = np.arange(len(device_category_counts))
axes[0].barh(y0, device_category_counts["count"].to_numpy())
axes[0].set_yticks(y0)
axes[0].set_yticklabels(
    device_category_counts["DEVICE_CATEGORY"].fillna("NULL").astype(str).tolist()
)
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Device Category")
axes[0].set_title("Activity by Device Category")
axes[0].invert_yaxis()

# Device OS
y1 = np.arange(len(device_os_counts))
axes[1].barh(y1, device_os_counts["count"].to_numpy())
axes[1].set_yticks(y1)
axes[1].set_yticklabels(
    device_os_counts["DEVICE_OPERATING_SYSTEM"].fillna("NULL").astype(str).tolist()
)
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Device OS")
axes[1].set_title("Activity by Device OS")
axes[1].invert_yaxis()

# Device brand
y2 = np.arange(len(device_brand_counts))
axes[2].barh(y2, device_brand_counts["count"].to_numpy())
axes[2].set_yticks(y2)
axes[2].set_yticklabels(
    device_brand_counts["DEVICE_MOBILE_BRAND_NAME"].fillna("NULL").astype(str).tolist()
)
axes[2].set_xlabel("Count")
axes[2].set_ylabel("Device Brand")
axes[2].set_title("Activity by Device Brand")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

The device and operating system data shows clear dominance patterns across users. Desktop users account for the largest share of activity compared to mobile and tablet devices. Since session identifiers are not available, it is not possible to control for differences in the amount of activity per session, which means desktop usage counts may be inflated if those users simply engage in more actions during a visit. A similar trend is seen in operating systems, where Windows dramatically exceeds all other platforms. Again, without session-level detail it is unclear how much of this lead is due to true user preference versus inflated counts from higher engagement. Within mobile devices, Google-based systems such as Android strongly dominate, with Apple devices in a distant second place. Even considering inflation, the wide margin suggests that Android users either make up a larger share of the Coke365 user base or are more active on the site. Although inflation reduces precision in interpretation, the overall chart remains helpful by showing that the platforms at the top either had more users or more usage intensity.

Observations:
- Desktop users easily dominate over the other two devices (mobile and tablet), though we don't have session id's and can't remove the effect the amount of activity per session on each device has on this count. If Desktop users do more on the website, this could could be inflated quite a bit.
- Windows also dramatically dominates the other operating systems. This could again be due to inflation, but since we don't have session ids or any way to break the data into sessions, we can't tell how much is inflated and how much the os really beats the others.
- Google mobile devices (such as Android) dominate the mobile landscape. The inflation will be here too, but there is still a wide margin between Google and Apple (the second place device), so either the device is far more popular with Coke365 users, or they are on the site doing more in general.
- The overall inflation, while limiting our precision, still shows that either the users at the top used the site more on their systems, or still had more users using them. Either way, the chart is still helpful.

In [0]:
import numpy as np
import matplotlib.pyplot as plt

# Top 10 event page/title/name
page_title_counts = (
    google_analytics["EVENT_PAGE_TITLE"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "EVENT_PAGE_TITLE", "EVENT_PAGE_TITLE": "count"})
    .head(10)
)

page_name_series = google_analytics["EVENT_PAGE_NAME"]
mask_not_null_literal = (
    (~page_name_series.astype(str).str.lower().eq("null")) & (page_name_series.notna())
)
page_name_counts = (
    page_name_series.where(mask_not_null_literal)
    .dropna()
    .value_counts()
    .reset_index()
    .rename(columns={"index": "EVENT_PAGE_NAME", "EVENT_PAGE_NAME": "count"})
    .head(10)
)

event_name_counts = (
    google_analytics["EVENT_NAME"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "EVENT_NAME", "EVENT_NAME": "count"})
    .head(10)
)

fig, axes = plt.subplots(1, 3, figsize=(12, 6))

# Page titles
y0 = np.arange(len(page_title_counts))
axes[0].barh(y0, page_title_counts["count"].to_numpy())
axes[0].set_yticks(y0)
axes[0].set_yticklabels(
    page_title_counts["EVENT_PAGE_TITLE"].fillna("NULL").astype(str).tolist()
)
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Event Page Title")
axes[0].set_title("Top Event Page Titles")
axes[0].invert_yaxis()

# Page names (already filtered of NaN and literal 'null', but coerce to string anyway)
y1 = np.arange(len(page_name_counts))
axes[1].barh(y1, page_name_counts["count"].to_numpy())
axes[1].set_yticks(y1)
axes[1].set_yticklabels(
    page_name_counts["EVENT_PAGE_NAME"].astype(str).tolist()
)
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Event Page Name")
axes[1].set_title("Top Event Page Names")
axes[1].invert_yaxis()

# Event names
y2 = np.arange(len(event_name_counts))
axes[2].barh(y2, event_name_counts["count"].to_numpy())
axes[2].set_yticks(y2)
axes[2].set_yticklabels(
    event_name_counts["EVENT_NAME"].fillna("NULL").astype(str).tolist()
)
axes[2].set_xlabel("Count")
axes[2].set_ylabel("Event Name")
axes[2].set_title("Top Event Names")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

The top event page titles indicate that users most often engage with product list pages. This is followed by frequent visits to the home screen, which is likely explained by its role as the default login page or as a common transition point between sections. Cart and search pages appear next in frequency, which reinforces their role in supporting the core shopping experience. Looking at the top event page names, the order page is the most visited, followed by the user’s dashboard where visit plans are likely managed, along with the cart and product order view. In terms of event names, browsing actions dominate, with button clicks following closely and cart management events appearing next. Together, these three perspectives suggest that users are actively engaging with the application in ways that reflect its intended purpose and goals.

Observations:
- The Top Event Page Titles make it seem like users are most often looking at prodoct lists. 
- This is followed by being on the home screen (likely due to that as the default page at login, or intermitently as a page transition).
- The cart and search pages follow that, promoting the primary use of the application.
- In the Top Event Page Names, it seems the order page is the most visited, followed by the user's dashboard (likely where they manage their visit plan), their cart and product order view.
- In the Top Event Names, it seems users most often browse, with button clicks coming next. This is followed by cart management events
- These three sections reflect healthy usage of the application, and a good emphasis on user engagement in the product use goal

## Sales

In [0]:
print(sales_raw.head())
summarize(sales_raw, name="sales_raw")

The order table requires several data type adjustments for proper analysis. POSTING_DATE should be cast to a date, and MATERIAL_ID should be cast to an integer. While identifiers are typically left as strings, testing shows this casting aligns with item identifiers in the Google Analytics table. Both GROSS_PROFIT_DEAD_NET and PHYSICAL_VOLUME should be cast to floats. Although quantities are often stored as integers, a float is preferable here to avoid potential issues if decimal values appear. Notably, the table contains no null values, which simplifies preparation.

The summary statistics reveal strong skew across several key measures. For GROSS_PROFIT_DEAD_NET, the mean is 71.26 while the median is 32.11, highlighting a substantial skew that suggests the median will be a more reliable measure unless the skew is explained by extreme outliers. NSI_DEAD_NET shows a similar pattern with a mean of 179.21 and a median of 76.68, again indicating that median values may provide a clearer representation of central tendency. PHYSICAL_VOLUME is also highly skewed with a mean of 6.78 and a median of 2.0, alongside a maximum of 99.00 and a minimum of 1.04. This distribution suggests heavy skew that could be driven by a small number of large orders. As with the other measures, median values will likely be the preferred reference point unless further analysis shows that outliers are driving the effect.

Observations:
- POSTING_DATE needs to be cast to a date.
- MATERIAL_ID needs to be cast to an integer (tested with a not padded value against items in the ITEMS column of the google_analytics table), though usually you wouldn't do this with an identifier in practice.
- GROSS_PROFIT_DEAD_NET needs to be cast to a float.
- PHYSICAL_VOLUME needs to be cast to a float. While quantities are often cast to a integer, using a float here ensures we aren't surprised later if there were any decimal numbers.
- This table has no null values.
- The mean and median of GROSS_PROFIT_DEAD_NET are $71.26 and $32.11 respectively, showing a pretty dramatic skew. Likely we will want to consider median for future analysis unless the skew is due to heavy outliers.
- The mean and median of NSI_DEAD_NET are $179.21 and $76.68 respectively, reflecting a similar skew. The median will be more important here as well unless outliers are the cause.
- The mean and median of PHYSICAL_VOLUME are 6.78 and 2.0 respectively. With a max of 99.00 and a min of 1.04 we can see a dramatically heavy skew. Median here as well, unless the effect is mostly due to outliers.

Visuals:
- Consider the two monetary values over time
- Check Skew with boxplots


In [0]:
sales = sales_raw.copy()
sales["POSTING_DATE"] = pd.to_datetime(sales["POSTING_DATE"], format="%m/%d/%Y", errors="coerce")
for col in ["GROSS_PROFIT_DEAD_NET", "PHYSICAL_VOLUME", "NSI_DEAD_NET"]:
    sales[col] = to_numeric_strip_commas(sales[col])

sales = sales[[
    "CUSTOMER_ID", "MATERIAL_ID", "POSTING_DATE",
    "GROSS_PROFIT_DEAD_NET", "NSI_DEAD_NET", "PHYSICAL_VOLUME"
]]

The raw sales table was cleaned and reformatted to prepare it for analysis. The POSTING_DATE column was converted into a proper date type. The GROSS_PROFIT_DEAD_NET, PHYSICAL_VOLUME, and NSI_DEAD_NET columns had commas removed from their string values and were then cast to doubles so they can be used in numeric calculations. Finally, the columns were reordered to keep like column types together.

In [0]:
# Monetary changes over time

In [0]:
# Skew analysis

## Order

In [0]:
print(orders_raw.head())

orders = orders_raw.copy()

# ORDER_TIMESTAMP_UTC: coalesce(CREATED_DATE_UTC, to_utc_timestamp(CREATED_DATE_EST, 'America/New_York'))
created_utc = pd.to_datetime(orders.get("CREATED_DATE_UTC"), errors="coerce", utc=True)
created_est = pd.to_datetime(orders.get("CREATED_DATE_EST"), errors="coerce")  # naive local EST
# Localize naive EST then convert to UTC
try:
    created_est_local = created_est.dt.tz_localize("America/New_York", nonexistent='NaT', ambiguous='NaT')
    created_est_utc = created_est_local.dt.tz_convert("UTC")
except Exception:
    created_est_utc = pd.Series(pd.NaT, index=orders.index)

orders["ORDER_TIMESTAMP_UTC"] = coalesce_series(created_utc, created_est_utc)

orders["ORDER_QUANTITY"] = to_numeric_strip_commas(orders["ORDER_QUANTITY"])

orders = orders[[
    "CUSTOMER_ID", "MATERIAL_ID", "PLANT_ID", "ORDER_TYPE",
    "ORDER_QUANTITY", "ORDER_TIMESTAMP_UTC"
]]

print(orders.head())
summarize(orders, name="orders")

## Customers

In [0]:
print(customers_raw.head())

customers = customers_raw.rename(columns={
    "SALES_OFFICE":"SALES_OFFICE_ID",
    "SALES_OFFICE_DESCRIPTION":"SALES_OFFICE_LOCATION",
    "DISTRIBUTION_MODE_DESCRIPTION":"DISTRIBUTION_MODE_DESC",
    "SHIPPING_CONDITIONS_DESCRIPTION":"SHIPPING_CONDITIONS_DESC",
    "COLD_DRINK_CHANNEL_DESCRIPTION":"COLD_DRINK_CHANNEL_DESC",
    "CUSTOMER_SUB_TRADE_CHANNEL_DESCRIPTION":"CUSTOMER_SUB_TRADE_CHANNEL_DESC"
})[[
    "CUSTOMER_NUMBER", "SALES_OFFICE_ID", "SALES_OFFICE_LOCATION",
    "DISTRIBUTION_MODE_DESC", "SHIPPING_CONDITIONS_DESC",
    "COLD_DRINK_CHANNEL_DESC", "CUSTOMER_SUB_TRADE_CHANNEL_DESC"
]]

print(customers.head())
summarize(customers, name="customers")

customer_routing = customers[["CUSTOMER_NUMBER","SALES_OFFICE_ID"]].copy()

## Materials

In [0]:
print(materials_raw.head())
materials = materials_raw.copy()
summarize(materials, name="materials")

## Operating Hours

In [0]:
print(operating_hours_raw.head())
operating_hours = operating_hours_raw.copy()
summarize(operating_hours, name="operating_hours")

## Visit Plan

In [0]:
print(visit_plan_raw.head())

# Frequency maps
frequency_map_days = {
    "00": 0,
    "01": 7, "02": 14, "03": 21, "04": 28, "05": 35,
    "06": 42, "07": 49, "08": 56, "09": 63, "10": 70,
    "11": 77, "12": 84, "13": 91, "14": 98, "15": 105,
}

def normalize_null_like(s: pd.Series) -> pd.Series:
    s2 = s.copy()
    s2 = s2.where(~s2.astype(str).str.strip().isin(["", "null", "NULL"]), other=pd.NA)
    return s2

visit_plan = visit_plan_raw.copy()
visit_plan["ELT_TS"] = normalize_null_like(visit_plan.get("ELT_TS"))
visit_plan["SNAPSHOT_DATE"] = normalize_null_like(visit_plan.get("SNAPSHOT_DATE"))
visit_plan["ANCHOR_DATE"] = normalize_null_like(visit_plan.get("ANCHOR_DATE"))

visit_plan["ELT_TS_UTC"] = pd.to_datetime(visit_plan["ELT_TS"], errors="coerce", utc=True)
visit_plan["SNAPSHOT_DATE"] = pd.to_datetime(visit_plan["SNAPSHOT_DATE"], format="%Y-%m-%d", errors="coerce")
visit_plan["ANCHOR_DATE"] = pd.to_datetime(visit_plan["ANCHOR_DATE"], format="%Y-%m-%d", errors="coerce")
visit_plan["ANCHOR_DAY_OF_WEEK"] = visit_plan["ANCHOR_DATE"].dt.dayofweek.add(1)  # Spark dayofweek: 1..7

# SHIPPING_CONDITIONS from description containing 24/48/72
sc_desc = visit_plan.get("SHIPPING_CONDITIONS_DESC").astype(str)
visit_plan["SHIPPING_CONDITIONS"] = np.select(
    [sc_desc.str.contains("24", na=False), sc_desc.str.contains("48", na=False), sc_desc.str.contains("72", na=False)],
    ["24hrs", "48hrs", "72hrs"],
    default=pd.NA
)

# Frequency normalization: text or raw digits -> 2-char code
freq = visit_plan.get("FREQUENCY").astype(str).str.strip()
mapping_text_to_code = {
    "Every Week On":"01", "Every Second Week On":"02", "Every Third Week On":"03",
    "Every Fourth Week On":"04", "Every Fifth Week On":"05", "Every Sixth Week On":"06",
    "Every Seventh Week On":"07", "Every Eighth Week On":"08", "Every Ninth Week On":"09",
    "Every Tenth Week On":"10", "Every Eleventh Week On":"11", "Every Twelfth Week On":"12",
    "Every Thirteenth Week On":"13", "Every Fourteenth Week On":"14", "Every Fifteenth Week On":"15",
    "Not Applicable":"00", "null":"00", "NULL":"00", "": "00"
}
# If it's just a digit 1..15, left-pad to 2
freq_code = freq.replace(mapping_text_to_code)
freq_code = freq_code.where(~freq_code.str.fullmatch(r"\d{1,2}"), other=freq_code.str.zfill(2))
# Anything else -> leave as-is
visit_plan["FREQUENCY"] = freq_code

visit_plan["FREQUENCY_DAYS"] = visit_plan["FREQUENCY"].map(frequency_map_days)

# Select/rename
visit_plan = visit_plan.rename(columns={
    "SALES_OFFICE":"SALES_OFFICE_ID",
    "SALES_OFFICE_DESC":"SALES_OFFICE_LOCATION"
})[[
    "CUSTOMER_ID","FREQUENCY","FREQUENCY_DAYS","ELT_TS_UTC","SNAPSHOT_DATE",
    "ANCHOR_DATE","ANCHOR_DAY_OF_WEEK","SHIPPING_CONDITIONS",
    "SALES_OFFICE_ID","SALES_OFFICE_LOCATION","DISTRIBUTION_MODE","SHIPPING_CONDITIONS_DESC"
]]

print(visit_plan.head())
summarize(visit_plan, name="visit_plan")

## Cutoff Times

In [0]:
print(cutoff_times_raw.head())

mode_map = {
    "OFS": "OF",
    "Rapid Delivery": "RD",
    "E Pallet": "EZ",
    "Sideload": "SL",
    "Night Sideload": "NS",
    "Full Service": "FS",
    "Night Rapid Delivery": "NR",
    "Night OFS": "NO",
    "Special Events": "SE",
    "Bulk Distribution": "BK"
}

cutoff_times = cutoff_times_raw.copy().rename(columns={"DISTRIBUTION_MODE":"DISTRIBUTION_MODE_FULL"})
cutoff_times["DISTRIBUTION_MODE"] = cutoff_times["DISTRIBUTION_MODE_FULL"].map(mode_map)

# Filter: length checks and non-null distribution mode
def str_len_ge(series, n):
    return series.astype(str).str.len().ge(n)

mask_valid = str_len_ge(cutoff_times.get("SALES_OFFICE"), 2) & \
             str_len_ge(cutoff_times.get("PLANT_ID"), 2) & \
             cutoff_times["DISTRIBUTION_MODE"].notna()

cutoff_times = cutoff_times.loc[mask_valid, [
    "SALES_OFFICE", "PLANT_ID", "CUTOFFTIME__C", "SHIPPING_CONDITION_TIME",
    "DISTRIBUTION_MODE", "DISTRIBUTION_MODE_FULL"
]].rename(columns={
    "SALES_OFFICE":"SALES_OFFICE_LOCATION",
    "CUTOFFTIME__C":"CUTOFF_TIME",
    "SHIPPING_CONDITION_TIME":"SHIPPING_CONDITIONS"
})

print(cutoff_times.head())
summarize(cutoff_times, name="cutoff_times")

## Sales Office

In [0]:
state_tz = {
    "AL":"America/Chicago","AK":"America/Anchorage","AZ":"America/Phoenix",
    "AR":"America/Chicago","CA":"America/Los_Angeles","CO":"America/Denver",
    "CT":"America/New_York","DC":"America/New_York","DE":"America/New_York",
    "FL":"America/New_York","GA":"America/New_York","HI":"Pacific/Honolulu",
    "ID":"America/Denver","IL":"America/Chicago","IN":"America/Indiana/Indianapolis",
    "IA":"America/Chicago","KS":"America/Chicago","KY":"America/New_York",
    "LA":"America/Chicago","ME":"America/New_York","MD":"America/New_York",
    "MA":"America/New_York","MI":"America/Detroit","MN":"America/Chicago",
    "MS":"America/Chicago","MO":"America/Chicago","MT":"America/Denver",
    "NE":"America/Chicago","NV":"America/Los_Angeles","NH":"America/New_York",
    "NJ":"America/New_York","NM":"America/Denver","NY":"America/New_York",
    "NC":"America/New_York","ND":"America/Chicago","OH":"America/New_York",
    "OK":"America/Chicago","OR":"America/Los_Angeles","PA":"America/New_York",
    "RI":"America/New_York","SC":"America/New_York","SD":"America/Chicago",
    "TN":"America/Chicago","TX":"America/Chicago","UT":"America/Denver",
    "VT":"America/New_York","VA":"America/New_York","WA":"America/Los_Angeles",
    "WV":"America/New_York","WI":"America/Chicago","WY":"America/Denver"
}

sales_office_a = visit_plan[["SALES_OFFICE_ID","SALES_OFFICE_LOCATION"]].rename(columns={"SALES_OFFICE_LOCATION":"LOCATION"})
sales_office_b = customers[["SALES_OFFICE_ID","SALES_OFFICE_LOCATION"]].rename(columns={"SALES_OFFICE_LOCATION":"LOCATION"})
sales_office = pd.concat([sales_office_a, sales_office_b], ignore_index=True)

# Filter out nulls / literal 'null'
mask_valid_so = sales_office["SALES_OFFICE_ID"].notna() & ~sales_office["SALES_OFFICE_ID"].astype(str).str.lower().eq("null")
sales_office = sales_office.loc[mask_valid_so].drop_duplicates().sort_values("SALES_OFFICE_ID")

# Extract trailing 2-letter state
sales_office["STATE"] = sales_office["LOCATION"].astype(str).str.upper().str.extract(r"([A-Z]{2})$")
sales_office["TIMEZONE"] = sales_office["STATE"].map(state_tz)

print(sales_office.head())

# Join TIMEZONE onto visit_plan and customers by SALES_OFFICE_ID
visit_plan = visit_plan.merge(
    sales_office[["SALES_OFFICE_ID","TIMEZONE"]],
    on="SALES_OFFICE_ID", how="left"
)
customers = customers.merge(
    sales_office[["SALES_OFFICE_ID","TIMEZONE"]],
    on="SALES_OFFICE_ID", how="left"
)

# Remodeling Tables

In [0]:
order_policy = visit_plan.merge(
    cutoff_times,
    on=["SALES_OFFICE_LOCATION","SHIPPING_CONDITIONS","DISTRIBUTION_MODE"],
    how="left"
)

# Show rows where FREQUENCY_DAYS is null (like the Spark display(filter(...isNull())))
mask_freq_null = order_policy["FREQUENCY_DAYS"].isna()
print("\nRows where FREQUENCY_DAYS is null:\n", order_policy.loc[mask_freq_null].head())